In [ ]:
#importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.nodel_selection import train_test_split
from sklearn.ensemble import RandonForestClassifier
from sklearn.nultioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

from sklearn.preprocessing import MinMaxScaler
import joblib

In [ ]:
#load and processing data
df = pd.read_csv("irrigation_machine.csv")

In [ ]:
#Display first few rows
print("Dataset preview")
print(df.head())

In [ ]:
df.info()

In [3]:
df.columns

NameError: name 'df' is not defined

In [ ]:
df = df.drop('Unnamed: 0',axis=1)
df.head()

In [ ]:
df.describe()#Statistics of the dataset

In [ ]:
#STEP2:define features and lab
x=df.iloc[:,0:20]
y = df.iloc[:,20:]

In [ ]:
x.sample(10)

In [ ]:
y.sample(10)

In [ ]:
x.info()

In [ ]:
y.info()

In [ ]:
x.shape,y.shape

In [ ]:
scaler = MinMaxScaler()
x_scaled = scaler.fit_tranform(x)
x_scaled

In [ ]:
#STEP 3: train-test split
x_train, x_test, y_train = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
#STEP 4: train classifier
#Use multioutputClassifier to handle multi-label classification
from sklearn.ensemble import  RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

#Custom hyperparameters for RandomForest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

#Wrap it with MultiOutputClassifier
model = MultiOutputClassifier(rf)
#Train the model
model.fit(x_train, y_train)

In [ ]:
#STEP 5: Evaluate model
y_pred = model.predict(x_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=y.columns))

In [ ]:
print(df(['parcel_0','parcel_1','parcel_2']),sum())

In [ ]:
import matplotlib.pyplot as plt

#Define parcel activation conditions with descriptive labels
conditions = {
    "Parcel 0 ON":df['parcel_0'],
    "Parcel 1 ON":df['parcel_1'],
    "Parcel 2 ON":df['parcel_2'],
    "Parcel 0 & 1 ON":df['parcel_0'] & df['parcel_1'],
    "Parcel 0 & 2 ON":df['parcel_0'] & df['parcel_2'],
    "Parcel 1 & 2 ON":df['parcel_1'] & df['parcel_2'],
    "All Parcels ON":df['parcel_0'] & df['parcel_1'] & df['parcel_2']
}

#create vertically stacked subplots(one for each condition)
fig, axs = plt.subplots(nrows=len(conditions), figsize=(10,15), sharex=True)

#Loop through each condition to plot corresponding square wave
for ax, (title, condition) in zip(axs, conditions.item()):
    ax.step(df.index, condition.astype(int), where'post', linewidth=1, color='teal')
    ax.set_title(f"sprinkler - {title}")
    ax.set_ylabel("Status")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['OFF', 'ON'])

    #Label x-axis on the last subplot
    axs[-1].set_xlabel("Time Index(Row Number)")

    #Plot
    plt.show()

In [ ]:
#Calculate combined activity of all pumps(overlap)
any_pump_on = (df['parcel_0'] == 1) | (df['parcel_1'] == 1) | (df['parcel_2'] == 1)
plt.figure(figsize=(15,5))

#Plot individual pump statuses
plt.step(df.index, df['parcel_0'], where='post', linewidth=2, label='Parcel 0 Pump', color='blue')
plt.step(df.index, df['parcel_1'], where='post', linewidth=2, label='Parcel 1 Pump', color='orange')
plt.step(df.index, df['parcel_2'], where='post', linewidth=2, label='Parcel 2 Pump', color='green')

plt.title("Pump Activity and combined Fram coverage")
plt.xlabel("Time Index (Row Number)")
plt.ylabel("Status")
plt.yticks([0, 1],['OFF', 'ON'])
plt.legend(loc='upper right')
plt.show()

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

joblib.dump(model, "Farm_Irrigation_system.pkl")